In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
file_name = "C:/user/admin/Downloads/meetinglistdetails_2026_05_29_2026_05_29.csv"
file_extension = os.path.splitext(file_name)[1].lower()

# Track if the file loaded successfully
file_loaded = False
df_raw = None

try:
    if file_extension == '.csv':
        df_raw = pd.read_csv(file_name)
        print("Successfully loaded CSV dataset!")
        file_loaded = True
    elif file_extension in ['.xlsx', '.xls']:
        df_raw = pd.read_excel(file_name)
        print("Successfully loaded Excel dataset!")
        file_loaded = True
    else:
        print(f"[ERROR]: Unsupported file format: {file_extension}. Please use a .csv or .xlsx file.")

except FileNotFoundError:
    print(f"\n[ERROR]: The system could not find the file at: {file_name}")
    print("Please double-check your file path, file name, and spelling.")
except Exception as e:
    print(f"\n[ERROR]: An unexpected error occurred while loading the file:\n{e}")

# Only proceed with data processing if the file was successfully loaded
if file_loaded and df_raw is not None:
    
    student_pool = df_raw.groupby('Email')['Name (original name)'].first().reset_index()
    
    # Safeguard: Sample 600 if available, otherwise take the total pool size
    sample_size = min(600, len(student_pool))
    df_students = student_pool.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    np.random.seed(42)
    num_days = 50
    day_columns = [f"Day_{i}" for i in range(1, num_days + 1)]
    attendance_matrix = []
    
    for idx in range(len(df_students)):
        if idx % 3 == 0:
            p_present = np.random.uniform(0.82, 0.98) 
        elif idx % 3 == 1:
            p_present = np.random.uniform(0.72, 0.84) 
        else:
            p_present = np.random.uniform(0.50, 0.78)    
        student_attendance = np.random.choice([1, 0], size=num_days, p=[p_present, 1 - p_present])
        attendance_matrix.append(student_attendance)
        
    df_days = pd.DataFrame(attendance_matrix, columns=day_columns)
    df_system = pd.concat([df_students, df_days], axis=1)
    df_system['Total_Present'] = df_system[day_columns].sum(axis=1)
    df_system['Attendance_Percentage'] = (df_system['Total_Present'] / num_days) * 100
    df_system['Eligible'] = np.where(df_system['Attendance_Percentage'] >= 80, 1, 0)
    
    X = df_system[day_columns]
    y = df_system['Eligible']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    ml_model = RandomForestClassifier(n_estimators=100, random_state=42)
    ml_model.fit(X_train, y_train)
    y_pred = ml_model.predict(X_test)
    
    eligible_list = df_system[df_system['Eligible'] == 1][['Name (original name)', 'Email', 'Attendance_Percentage']].copy()
    not_eligible_list = df_system[df_system['Eligible'] == 0][['Name (original name)', 'Email', 'Attendance_Percentage']].copy()
    
    eligible_list = eligible_list.sort_values(by='Attendance_Percentage', ascending=False)
    not_eligible_list = not_eligible_list.sort_values(by='Attendance_Percentage', ascending=False)
    
    excel_output_path = "Attendance_Final_Report1.xlsx"
    with pd.ExcelWriter(excel_output_path, engine='openpyxl') as writer:
        eligible_list.to_excel(writer, sheet_name='Eligible Students', index=False)
        not_eligible_list.to_excel(writer, sheet_name='Not Eligible Students', index=False)
        
    print(f"\nSuccessfully processed {len(df_system)} students.")
    print(f"-> Excel workbook generated: '{excel_output_path}'")
    print(f"   - Sheet 1: 'Eligible Students' ({len(eligible_list)} members)")
    print(f"   - Sheet 2: 'Not Eligible Students' ({len(not_eligible_list)} members)\n")
    
    print("==========================================================")
    print(f"   SAMPLE VIEW: ELIGIBLE STUDENTS (Total: {len(eligible_list)})")
    print("==========================================================")
    print(eligible_list.head(5))
    
    print("\n==========================================================")
    print(f"   SAMPLE VIEW: NOT ELIGIBLE STUDENTS (Total: {len(not_eligible_list)})")
    print("==========================================================")
    print(not_eligible_list.head(5))


[ERROR]: The system could not find the file at: C:/user/admin/Downloads/meetinglistdetails_2026_05_29_2026_05_29.csv
Please double-check your file path, file name, and spelling.
